# Get Synthetic Biology articles from OpenAlex

Downloads academic articles matching a keyword query from the [OpenAlex API](https://docs.openalex.org/),
flattens nested fields into plain strings suitable for tab-separated storage,
and saves the result as a `.txt` (TSV) file in `../assets/openalex_data/`.

In [ ]:
# ── QUERY CONFIG ──────────────────────────────────────────────────────────────
# Queries are defined in query.yaml (split into parts due to OpenAlex length limits).
# The notebook loops through ALL parts automatically — no manual re-runs needed.

import yaml

with open('query.yaml') as f:
    config = yaml.safe_load(f)

YEAR_START = config['year_start']
YEAR_END   = config['year_end']

print(f"Loaded {len(config['parts'])} query parts  |  Years: {YEAR_START}–{YEAR_END}")
for p in config['parts']:
    print(f"  {p['name']}: {p['query'][:80].strip()}…")

In [ ]:
import os
import time
import random
import requests
import pandas as pd

In [ ]:
OUTPUT_DIR = os.path.join('..', 'assets', 'openalex_data')
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Query helpers

In [ ]:
WORKS_ENDPOINT = 'https://api.openalex.org/works'

# Fields to retrieve from OpenAlex
SELECT_FIELDS = ','.join([
    'id', 'doi', 'title', 'publication_year', 'language', 'type',
    'primary_location', 'authorships', 'cited_by_count',
    'referenced_works', 'abstract_inverted_index', 'concepts'
])


def build_query_url(keywords: str, year_start: int, year_end: int) -> str:
    """Build the full OpenAlex API URL for a keyword + year-range search."""
    filters = [
        f'title_and_abstract.search:{keywords.strip()}',
        f'from_publication_date:{year_start}-01-01',
        f'to_publication_date:{year_end}-12-31',
        'type:article',
        'is_retracted:false',
    ]
    email = 'mailto=mejia.cristian@ifi.u-tokyo.ac.jp'
    return (
        f'{WORKS_ENDPOINT}'
        f'?filter={",".join(filters)}'
        f'&sort=cited_by_count:desc'
        f'&select={SELECT_FIELDS}'
        f'&{email}'
    )

## 2. Download

In [ ]:
def download_all(query_url: str, per_page: int = 200, pause: float = 1.0) -> list[dict]:
    """Page through the full result set using cursor pagination."""
    cursor = '*'
    records: list[dict] = []
    while cursor is not None:
        resp = requests.get(f'{query_url}&per-page={per_page}&cursor={cursor}')
        resp.raise_for_status()
        data = resp.json()
        meta = data['meta']
        print(f"  fetched {len(records) + len(data['results']):>6,} / {meta['count']:,}", end='\r')
        records.extend(data['results'])
        cursor = meta.get('next_cursor')
        time.sleep(max(0.5, pause))
    print(f"  done — {len(records):,} records downloaded.")
    return records

## 3. Flatten nested fields for TSV storage

OpenAlex returns several nested objects (inverted-index abstracts, authorships, concepts, etc.).
We flatten them into plain strings so every column is scalar and the data can be stored as a `.txt` tab-separated file.

In [ ]:
# Load ISO-3166 country code mapping once
_country_codes = pd.read_csv(
    'https://raw.githubusercontent.com/cristianmejia00/display/refs/heads/main/ISO3166_country_codes.csv'
)
_ISO_COUNTRY = dict(zip(_country_codes['alpha_2_code'], _country_codes['country']))

In [ ]:
def _invert_abstract(inv_index: dict | None) -> str:
    """Reconstruct plain-text abstract from OpenAlex inverted index."""
    if not inv_index:
        return ''
    pairs = [(word, pos) for word, positions in inv_index.items() for pos in positions]
    pairs.sort(key=lambda x: x[1])
    return ' '.join(w for w, _ in pairs)


def _flatten_concepts(concepts: list[dict], threshold: float = 0.3) -> str:
    """Join concept names with score >= threshold."""
    if not concepts:
        return ''
    return '; '.join(c['display_name'] for c in concepts if c.get('score', 0) >= threshold)


def _flatten_authorships(authorships: list[dict]) -> tuple[str, str, str]:
    """Return (authors, institutions, countries) as semicolon-separated strings."""
    authors, institutions, countries = [], set(), set()
    for entry in (authorships or []):
        name = entry.get('author', {}).get('display_name')
        if name:
            authors.append(name)
        for inst in entry.get('institutions', []):
            disp = inst.get('display_name')
            if disp:
                institutions.add(disp)
            cc = inst.get('country_code')
            if cc:
                countries.add(_ISO_COUNTRY.get(cc.upper(), cc))
    return '; '.join(authors), '; '.join(sorted(institutions)), '; '.join(sorted(countries))


def _flatten_source(primary_location: dict | None) -> tuple[str, str]:
    """Extract (source_id, source_name) from primary_location."""
    if primary_location and isinstance(primary_location, dict):
        src = primary_location.get('source') or {}
        return src.get('id', ''), src.get('display_name', '')
    return '', ''


def _flatten_refs(refs: list | None) -> str:
    """Join referenced-work IDs."""
    if not refs:
        return ''
    return '; '.join(refs)

In [ ]:
def records_to_dataframe(records: list[dict]) -> pd.DataFrame:
    """Convert raw OpenAlex records into a flat DataFrame ready for TSV export."""
    rows = []
    for r in records:
        authors, institutions, countries = _flatten_authorships(r.get('authorships'))
        source_id, source_name = _flatten_source(r.get('primary_location'))
        rows.append({
            'id':               r.get('id', ''),
            'doi':              r.get('doi', ''),
            'title':            r.get('title', ''),
            'publication_year': r.get('publication_year', ''),
            'language':         r.get('language', ''),
            'type':             r.get('type', ''),
            'source_id':        source_id,
            'source_name':      source_name,
            'authors':          authors,
            'institutions':     institutions,
            'countries':        countries,
            'cited_by_count':   r.get('cited_by_count', 0),
            'abstract':         _invert_abstract(r.get('abstract_inverted_index')),
            'concepts':         _flatten_concepts(r.get('concepts')),
            'referenced_works': _flatten_refs(r.get('referenced_works')),
        })
    df = pd.DataFrame(rows)
    df.drop_duplicates(subset='id', inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

## 4. Execute: download all parts, transform, and save

Loops through every part defined in `query.yaml`. Each part produces its own
`synbio_openalex_PART_x.txt` file in the output folder.

In [ ]:
for part in config['parts']:
    name  = part['name']
    query = part['query']
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    # Build URL and check count
    url = build_query_url(query, YEAR_START, YEAR_END)
    resp = requests.get(url)
    resp.raise_for_status()
    total = resp.json()['meta']['count']
    print(f"  Articles matching query: {total:,}")

    # Download
    raw_records = download_all(url)

    # Transform
    pubs = records_to_dataframe(raw_records)
    print(f"  Unique publications: {len(pubs):,}")

    # Save
    output_path = os.path.join(OUTPUT_DIR, f'synbio_openalex_{name}.txt')
    pubs.to_csv(output_path, sep='\t', index=False)
    print(f"  Saved → {output_path}")

print(f"\nAll {len(config['parts'])} parts complete.")